In [0]:
from pyspark.sql.functions import col, count, date_format, md5, concat_ws, first

# 1. Carregamento dos dados da camada Silver
df_silver = spark.read.table("silver_dengue_consolidado")

# 2. Criação da Dimensão Tempo
gold_dim_tempo = df_silver.select("data_notificacao").distinct() \
    .withColumn("id_tempo", date_format(col("data_notificacao"), "yyyyMMdd").cast("int"))

# 3. Criação da Dimensão Município (ID via MD5 do código)
# Atualizado para evitar duplicatas por valores nulos usando groupBy e first
gold_dim_municipio = df_silver.groupBy("codigo_municipio").agg(
    first("municipio", ignorenulls=True).alias("municipio"),
    first("sg_uf_not", ignorenulls=True).alias("sg_uf_not")
).withColumn("id_municipio", md5(col("codigo_municipio").cast("string")))

# 4. Criação da Dimensão Paciente (ID via MD5 de atributos demográficos)
gold_dim_paciente = df_silver.select("sexo", "raca", "gestante", "ano_nascimento").distinct() \
    .withColumn("id_paciente", md5(concat_ws("-", col("sexo"), col("raca"), col("gestante"), col("ano_nascimento"))))

# 5. Criação da Dimensão Gravidade (ID via MD5 da classificação)
gold_dim_gravidade = df_silver.select("classificacao_final").distinct() \
    .withColumn("id_gravidade", md5(col("classificacao_final").cast("string")))

# 6. Criação da Tabela Fato
gold_fato_casos = df_silver \
    .join(gold_dim_tempo, "data_notificacao") \
    .join(gold_dim_municipio, ["codigo_municipio", "municipio", "sg_uf_not"]) \
    .join(gold_dim_paciente, ["sexo", "raca", "gestante", "ano_nascimento"]) \
    .join(gold_dim_gravidade, "classificacao_final") \
    .groupBy("id_tempo", "id_municipio", "id_paciente", "id_gravidade") \
    .agg(count("*").alias("quantidade_casos"))

# 7. Salvamento das tabelas
tabelas_gold = {
    "gold_dim_tempo": gold_dim_tempo,
    "gold_dim_municipio": gold_dim_municipio,
    "gold_dim_paciente": gold_dim_paciente,
    "gold_dim_gravidade": gold_dim_gravidade,
    "gold_fato_casos": gold_fato_casos
}

for nome_tabela, df_gold in tabelas_gold.items():
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome_tabela)

print("Camada Gold (Star Schema) processada com sucesso!")